In [3]:
import random, uuid

customers_int = 10
ts_step_int = 1

class OrderGenerator:
    def __init__(self):
        self.customer_id_int = 0
        #
        self.ts_int = 0
        self.ts_step_int = ts_step_int

    def generate(self):
        order_id_str = str(uuid.uuid4())
        message_dict = {
            "key": order_id_str,
            "value": {"order_id": order_id_str,
                      "customer_id": random.randint(0, customers_int - 1),
                      "price": random.randint(1, 10000) / 100,
                      "ts": self.ts_int},
        }
        #
        self.ts_int += self.ts_step_int
        #
        return message_dict


In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

#

order_source_str = "orders"
sink_str = "orders_tumbling"
#
size_int = ts_step_int * 100
allowed_lateness_int = 1 * size_int
#
order_source_tn = Tn.source(order_source_str)
order_tn = (
    order_source_tn
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .tumbling_retention(lambda x: x["ts"], size_int, allowed_lateness_int)
    .distinct()
)
#
window_tn = order_tn.tumbling(size_int,
                              lambda x: x["ts"],
                              lambda x: x["customer_id"],
                              lambda agg, x, w: {"orders": agg["orders"] + w,
                                                 "total_price": agg["total_price"] + x["price"] * w},
                              {"orders": 0, "total_price": 0},
                              lambda by, agg: {"customer_id": by,
                                               "orders": agg["orders"],
                                               "total_price": agg["total_price"]})._peek("trigger")
#
built_tn = Tn.build(window_tn.sink(sink_str))


In [ ]:
built_tn.reset()

def push(customer_id, price, ts):
    built_tn.push(order_source_str, [{'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}])
    built_tn.latest()

# tumbling_int=100

print("=== Window [0,100) running ===")
push(1, 100, 10)  # window_end=100
push(1, 200, 50)  # window_end=100, same bucket
push(2, 150, 70)  # window_end=100, customer 2

print("\n=== Window [0,100) closing → Emit ===")
push(1, 300, 105) # watermark=105 > 100 → both customers fire!
                  # customer 1: orders=2, total=300
                  # customer 2: orders=1, total=150

print("\n=== Window [100,200) running ===")
push(2, 200, 120) # window_end=200
push(1, 100, 150) # window_end=200
push(2, 300, 180) # window_end=200

print("\n=== Late Arrival for window [0,100) – still inside allowed lateness ===")
push(1, 500, 90)  # ts=90 → window_end=100, watermark=180 < 200 → correction!
                  # customer 1: +{orders=3, total=800} / -{orders=2, total=300}

print("\n=== Window [100,200) closing → Emit ===")
push(1, 100, 205) # watermark=205 > 200 → both customers fire!
                  # customer 1: orders=2, total=400
                  # customer 2: orders=2, total=500

In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "orders_hopping"
#
size_int = ts_step_int * 100
hop_int = size_int // 2
allowed_lateness_int = size_int * 1
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .hopping_retention(lambda x: x["ts"], size_int, hop_int, allowed_lateness_int)
    .distinct()
)
#
window_tn = order_tn.hopping(size_int,
                             hop_int,
                             lambda x: x["ts"],
                             lambda x: x["customer_id"],
                             lambda agg, x, w: {"orders": agg["orders"] + w,
                                                "total_price": agg["total_price"] + x["price"] * w},
                             {"orders": 0, "total_price": 0},
                             lambda by, agg: {"customer_id": by,
                                              "orders": agg["orders"],
                                              "total_price": agg["total_price"]})._peek("trigger")
#
built_tn = Tn.build(window_tn.sink(sink_str))

In [ ]:
built_tn.reset()

def push(customer_id, price, ts, w=1):
    built_tn.push(order_source_str, [({'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}, w)])
    built_tn.latest()

# Konfiguration im Hinterkopf: tumbling_int=100, hop_int=50
# allowed_lateness_ms = 150
# Fenster 100: [0, 100)
# Fenster 150: [50, 150)
# Fenster 200: [100, 200)

print("=== 1. Normale Events für das erste Fenster ===")
push(1, 100, 10)  # In Fenster 100
push(1, 200, 60)  # In Fenster 100 UND Fenster 150
# Erwartung: Kein Output (Gatekeeper blockiert, max_ts = 60)

print("\n=== 2. Trigger Event -> Schließt Fenster 100 ===")
push(1, 300, 105) # max_ts = 105 -> Schließt Fenster 100 (da 105 >= 100)
                  # In Fenster 150 UND Fenster 200
# Erwartung: Emit für Fenster 100 (orders: 2, price: 300 [10 + 60])

print("\n=== 3. Erlaubte Lateness (In-Order für Fenster 150, aber LATE für Fenster 100) ===")
# Fenster 100 ist schon emittiert, aber der State ist wegen .expire() noch offen!
push(1, 50, 40)   # Fällt in Fenster 100
# Erwartung: Sofortiges Delta-Update/Korrektur für das bereits geschlossene Fenster 100!
# Output Fenster 100: orders: 3, price: 350

print("\n=== 4. Trigger Event -> Schließt Fenster 150 ===")
push(1, 400, 155) # max_ts = 155 -> Schließt Fenster 150 (155 >= 150)
                  # In Fenster 200 UND Fenster 250
# Erwartung: Emit für Fenster 150 (orders: 3, price: 550 [60 + 105 + 400])

print("\n=== 5. Native Retraction (Kunde storniert ein Event) ===")
# Wir ziehen das Event von vorhin (ts=105, price=300) mit Gewicht w=-1 ab!
push(1, 300, 105, w=-1)
# Erwartung: Da Fenster 150 und 200 betroffen sind:
# - Fenster 150 (bereits offen) emittiert ein Korrektur-Update (total_price sinkt um 300)
# - Im Zustand für Fenster 200 wird es direkt abgezogen (bevor es je emittiert wurde!)

print("\n=== 6. Hart an der Grenze: Letzte erlaubte Lateness ===")
# Für Fenster 100 ist der Expire-Cutoff bei ts=250. Wir senden ts=30 (gehört in 100)
push(1, 99, 30)
# Erwartung: Da max_ts noch 155 ist, schlägt das .expire() noch nicht zu.
# Fenster 100 kriegt ein weiteres Update!

print("\n=== 7. Der große Uhr-Sprung -> Auslösung & Expire-Grenze überschreiten ===")
push(1, 100, 255) # max_ts springt auf 255!
                  # Schließt Fenster 200 & 250.
                  # Löst zeitgleich das .expire() für alle Events < 100 aus!
# Erwartung: 
# - Emit für Fenster 200 (ohne das stornierte ts=105 Event!)
# - Der State säubert sich im Hintergrund für die alten Epochen.

print("\n=== 8. TOO LATE ARRIVAL (Gnadenlos verworfen) ===")
# Wir versuchen JETZT nochmal ein Event für das allererste Fenster zu senden
push(1, 999, 20)  # Gehört in Fenster 100

# Erwartung: KEIN Output. Das Event wird vom .expire() oben im Graphen gefiltert,
# da max_ts (255) > 20 + 100 + 150 (Cutoff) ist. Der Zustand existiert nicht mehr.

In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "orders_cumulative"
#
size_int = ts_step_int * 100
advance_int = size_int // 5
allowed_lateness_int = 0
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .cumulative_retention(lambda x: x["ts"], size_int, advance_int, allowed_lateness_int)
    .distinct()
)
#
window_tn = order_tn.cumulative(size_int,
                                advance_int,
                                lambda x: x["ts"],
                                lambda x: x["customer_id"],
                                lambda agg, x, w: {"orders": agg["orders"] + w,
                                                   "total_price": agg["total_price"] + x["price"] * w},
                                {"orders": 0, "total_price": 0},
                                lambda by, agg: {"customer_id": by,
                                                 "orders": agg["orders"],
                                                 "total_price": agg["total_price"]})._peek("trigger")
#
built_tn = Tn.build(window_tn.sink(sink_str))


In [ ]:
built_tn.reset()

def push(customer_id, price, ts, w=1):
    built_tn.push(order_source_str, [({'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}, w)])
    built_tn.latest()

print("=== 1. Erstes Event im ersten Slice ===")
push(1, 100, 10)  
# assign_cumulative -> [20, 40, 60, 80, 100]
# Erwartung: Kein Output (max_ts = 10 < 20)

print("\n=== 2. Event im zweiten Slice -> Löst das 20er-Fenster aus ===")
push(1, 50, 25)   
# assign_cumulative -> [40, 60, 80, 100]
# max_ts = 25 -> Löst Fenster 20 aus!
# Erwartung: Emit für window_end=20 (orders: 1, total_price: 100 [nur das ts=10 Event])

print("\n=== 3. Weiteres Event im selben Makro-Fenster -> Löst das 40er-Fenster aus ===")
push(1, 200, 45)  
# assign_cumulative -> [60, 80, 100]
# max_ts = 45 -> Löst Fenster 40 aus!
# Erwartung: Emit für window_end=40 (orders: 2, total_price: 150 [ts=10 + ts=25])

print("\n=== 4. Late Arrival (Kommt bei max_ts=45 an, gehört aber zu ts=15) ===")
# Das Event landet rückwirkend in [20, 40, 60, 80, 100]
push(1, 10, 15)   
# Erwartung: Sofortige Delta-Updates für die bereits emittierten Fenster 20 und 40!
# - Emit für window_end=20: orders: 2, total_price: 110
# - Emit für window_end=40: orders: 3, total_price: 160

print("\n=== 5. Native Retraction (Storno eines alten Events während des Tages) ===")
# Wir stornieren das Event von ts=25 (Wert 50) mit w=-1
push(1, 50, 25, w=-1)
# Erwartung: Korrektur-Updates für alle betroffenen Fenster, in denen es kumuliert war!
# - Fenster 20 ist NICHT betroffen (da ts=25 dort nie drin war).
# - Fenster 40 zieht die 50 ab -> orders: 2, total_price: 110.
# - In den noch offenen Zuständen (60, 80, 100) korrigiert es sich lautlos im Vorfeld.

print("\n=== 6. Großer Zeitsprung -> Löst kaskadenartig restliche Fenster aus ===")
push(1, 500, 95)  
# assign_cumulative -> [100]
# max_ts springt auf 95 -> Löst Fenster 60 und 80 zeitgleich aus!
# Erwartung:
# - Emit für window_end=60 (Kumulierter Wert bis ts=60 ohne das stornierte Event)
# - Emit für window_end=80 (Gleicher kumulierter Wert, da zwischen 60 und 80 nichts passierte)

print("\n=== 7. Sprung ins NÄCHSTE Makro-Fenster -> Schließt das alte Großfenster & triggert Expire ===")
push(1, 1000, 105) 
# assign_cumulative -> [120, 140, 160, 180, 200] (Neuer Tag!)
# max_ts = 105 -> Schließt das finale Fenster 100 des Vortrags ab.
# Zeitgleich greift das .expire() für alle Daten des ersten Tages (< 100).
# Erwartung: 
# - Emit für window_end=100 (Inhalt: Alle gültigen Tagesumsätze inklusive des ts=95er Events)
# - Im Hintergrund säubert DBSP den Zustand des ersten Tages restlos.

print("\n=== 8. TOO LATE ARRIVAL (Vortag ist abgelaufen) ===")
push(1, 999, 45)  
# Erwartung: Kein Output. Das Event wird oben durch .expire() direkt verworfen, 
# da das Makro-Fenster 0-100 im Graphen bereits gelöscht wurde.


In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "orders_session"
#
ts_step_int = 1
gap_int = ts_step_int * 20
max_session_int = ts_step_int * 200
allowed_lateness_int = 0
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .session_retention(lambda x: x["ts"], max_session_int, allowed_lateness_int)
    .distinct()
)
#
window_tn = order_tn.session(gap_int,
                             lambda x: x["ts"], 
                             lambda x: x["customer_id"], 
                             lambda agg, x, w: {"orders": agg["orders"] + w,
                                                "total_price": agg["total_price"] + x["price"] * w},
                             {"orders": 0, "total_price": 0},
                             lambda by, agg: {"customer_id": by,
                                              "orders": agg["orders"],
                                              "total_price": agg["total_price"]})._peek("trigger")
#
built_tn = Tn.build(window_tn.sink(sink_str))


In [ ]:
built_tn.reset()

def push(customer_id, price, ts, w=1):
    built_tn.push(order_source_str, [({'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}, w)])
    built_tn.latest()

# gap=20, max_session=200

print("=== Session customer 1 läuft ===")
push(1, 100, 0)   # session_bucket=0
push(1, 200, 10)  # session_bucket=0, verlängert
push(1, 300, 15)  # session_bucket=0, verlängert

print("\n=== Session customer 2 läuft parallel ===")
push(2, 150, 5)   # session_bucket=0
push(2, 250, 18)  # session_bucket=0, verlängert

print("\n=== Gap customer 1 abgelaufen → Emit ===")
push(2, 100, 60)  # watermark=60 > 15+20=35 → customers 1 feuert! orders=3, total=600, und customer 2 auch: orders: 2, total 400

print("\n=== Gap customer 2 abgelaufen → Emit ===")
push(1, 100, 90)  # watermark=90 > 18+20=38 → customer 2 feuert! orders=1, total=100

print("\n=== Hard Reset bei max_session_int → neue session_bucket ===")
push(1, 500, 205) # session_bucket=200 → frischer state!
push(1, 500, 210) # session_bucket=200, verlängert

print("\n=== Gap neue Session customer 1 abgelaufen → Emit ===")
push(2, 50, 250)  # watermark=250 > 210+20=230 → customer 1 feuert! orders=2, total=1000

In [23]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

#

order_source_str = "orders"
sink_str = "orders_sliding"
#
size_int = ts_step_int * 10
allowed_lateness_int = 0 * size_int
#
order_source_tn = Tn.source(order_source_str)
order_tn = (
    order_source_tn
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .sliding_retention(lambda x: x["ts"], size_int, allowed_lateness_int)._peek("ret")
)
#
def bla(agg, x, w):
    print(agg, x, w)
    x = {"orders": agg["orders"] + 1,
     "total_price": agg["total_price"] + x["price"]}
    return x

window_tn = order_tn.sliding(size_int,
                             lambda x: x["ts"],
                             lambda x: x["customer_id"],
                             bla,
                             {"orders": 0, "total_price": 0},
                             lambda by, agg: {"customer_id": by,
                                              "orders": agg["orders"],
                                              "total_price": agg["total_price"]})._peek("trigger")
#
built_tn = Tn.build(window_tn.sink(sink_str))



In [24]:
# Setup: 10 Sekunden Sliding Window, keine Lateness (allowed_lateness_int=0)
# window_size = 10

built_tn.reset()

def push(customer_id, price, ts):
    built_tn.push(order_source_str, [{'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}])
    # latest() gibt uns die emittierten Änderungen (Deltas) aus
    built_tn.latest()

print("=== 1. Erstes Event bei ts=2 (Wert: 100) ===")
# Erwartung: Ein neues Aggregat von 100 wird erzeugt (+1).
# Das Event wird bis ts=12 (2 + 10) im Fenster aktiv bleiben.
push(1, 100, 2)  
# Output: (1, 100, 1)


print("\n=== 2. Zweites Event bei ts=5 (Wert: 200) ===")
# Erwartung: 
# - Das alte Aggregat (100) wird zurückgezogen (-1).
# - Das neue Aggregat (300) wird emittiert (+1).
# Beide Events (ts=2 und ts=5) sind aktiv, da ts=5 < 12 ist.
push(1, 200, 5)  
# Output: (1, 100, -1), (1, 300, 1)


print("\n=== 3. Erstes Event fliegt raus bei ts=12 (Wert: 300) ===")
# Erwartung: Wir pushen ein neues Event bei ts=12. 
# Da ts=12 genau 10 Sekunden nach dem ersten Event (ts=2) liegt, fliegt ts=2 JETZT raus.
# - Das alte Aggregat (300) wird zurückgezogen (-1).
# - Das neue Aggregat (300 + 300 = 600) wird emittiert (+1).
# (Das ts=5 Event mit 200 ist noch aktiv, das ts=2 mit 100 ist abgezogen)
push(1, 300, 12)  
# Output: (1, 300, -1), (1, 600, 1)


print("\n=== 4. Zweites Event fliegt raus bei ts=15 (Wert: 0) ===")
# Erwartung: Wir pushen ein "leeres" Event (oder einen Dummy) bei ts=15, nur um die Zeit vorzuspringen.
# Da ts=15 genau 10 Sekunden nach ts=5 liegt, fällt ts=5 (200) nun ebenfalls heraus.
# Übrig bleibt nur noch das Event bei ts=12 (300).
# - Das alte Aggregat (500) wird zurückgezogen (-1).
# - Das neue Aggregat (300) wird emittiert (+1).
push(1, 0, 15)  
# Output: (1, 600, -1), (1, 600, 1)


print("\n=== 5. Alles läuft leer bei ts=23 ===")
# Erwartung: Wir springen auf ts=23. Das Event von ts=12 (300) läuft ab.
# - Das alte Aggregat (300) wird zurückgezogen (-1).
# - Das neue Aggregat (0) wird emittiert (+1) oder der Key verschwindet völlig.
push(1, 0, 23)  
# Output: (1, 300, -1), (1, 0, 1) (bzw. Key-Löschung)

=== 1. Erstes Event bei ts=2 (Wert: 100) ===
ret: ({'customer_id': 1, 'price': 100, 'ts': 2}, 1)
{'orders': 0, 'total_price': 0} {'customer_id': 1, 'price': 100, 'ts': 2} 1
trigger: ({'customer_id': 1, 'orders': 1, 'total_price': 100}, 1)

=== 2. Zweites Event bei ts=5 (Wert: 200) ===
ret: ({'customer_id': 1, 'price': 200, 'ts': 5}, 1)
{'orders': 0, 'total_price': 0} {'customer_id': 1, 'price': 200, 'ts': 5} 1
{'orders': 1, 'total_price': 200} {'customer_id': 1, 'price': 100, 'ts': 2} 1
trigger: ({'customer_id': 1, 'orders': 2, 'total_price': 300}, 1)
trigger: ({'customer_id': 1, 'orders': 1, 'total_price': 100}, -1)

=== 3. Erstes Event fliegt raus bei ts=12 (Wert: 300) ===
ret: ({'customer_id': 1, 'price': 300, 'ts': 12}, 1)
ret: ({'customer_id': 1, 'price': 100, 'ts': 2}, -1)
{'orders': 0, 'total_price': 0} {'customer_id': 1, 'price': 300, 'ts': 12} 1
{'orders': 1, 'total_price': 300} {'customer_id': 1, 'price': 200, 'ts': 5} 1
trigger: ({'customer_id': 1, 'orders': 2, 'total_price'

In [ ]:
import cloudpickle

built_tn.reset()
order_generator = OrderGenerator()

state_size_kb_float_list = []
for i in range(100):
    print(f"Step {i + 1}")
    order_message_dict_list = [order_generator.generate() for _ in range(100)]
    built_tn.push(order_source_str, order_message_dict_list)
    built_tn.latest()
    state_size_kb_float = len(cloudpickle.dumps(built_tn)) / 1024
    state_size_kb_float_list.append(state_size_kb_float)
#
print(f"Last 10 state sizes (KB): {state_size_kb_float_list[-10:]}")
